## Step 1: 数据清洗与目标变量构建 (Data Cleaning & Target Variable Construction)

In [2]:
import os
import pandas as pd
import numpy as np
import torch

# 【硬件自适应】检测并配置计算设备 (RTX 4060 Laptop GPU)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"当前使用的计算设备: {device}")
if torch.cuda.is_available():
    print(f"显卡型号: {torch.cuda.get_device_name(0)}")
    torch.cuda.empty_cache() # 初始清理显存

# 1. 数据加载与合并
def load_and_merge_data(base_path, llm_path):
    # 读取原始数据
    df_base = pd.read_csv(base_path)
    df_llm = pd.read_csv(llm_path)
    
    # 将 timestamp 列转换为 datetime 格式
    df_base['timestamp'] = pd.to_datetime(df_base['timestamp'])
    df_llm['timestamp'] = pd.to_datetime(df_llm['timestamp'])
    
    # 强制按时间正序排列
    df_base = df_base.sort_values('timestamp').reset_index(drop=True)
    df_llm = df_llm.sort_values('timestamp').reset_index(drop=True)
    
    # 由于 df_llm 包含了 df_base 的列，为避免合并时产生大量 _x, _y 后缀列
    # 我们仅提取 df_llm 中独有的特征列与 timestamp 进行交集(inner)合并
    llm_specific_cols = ['timestamp', 'reasoning_text', 'sentiment_class', 'action_class', 'action_score']
    
    # 以 timestamp 为键进行 inner join，确保数据严格对齐
    df_merged = pd.merge(df_base, df_llm[llm_specific_cols], on='timestamp', how='inner')
    
    return df_merged

# 2. 缺失值处理与目标变量构建
def preprocess_and_build_target(df):
    # 【红线：防穿越】严格使用前向填充 (ffill) 处理缺失值，绝对禁用 bfill 和线性插值
    df = df.ffill()
    
    # 丢弃最开头无法前向填充的 NaN 行
    df = df.dropna(subset=['high', 'low']).reset_index(drop=True)
    
    # 获取最高价和最低价
    H_t = df['high']
    L_t = df['low']
    
    # 规避极端情况（如H_t == L_t导致V_P为0，取对数后变为-inf），加上极小值 epsilon
    epsilon = 1e-8 
    
    # 计算 Parkinson 极值波动率 V_P = [ln(H_t / L_t)]^2 / (4 * ln2)
    V_P = (np.log(H_t / L_t))**2 / (4 * np.log(2))
    
    # 计算对数波动率 sigma_t = ln(V_P)
    # 使用 np.where 防止 V_P 恰好为 0 时产生 -inf，若存在此情况替换为 NaN 稍后删除
    sigma_t = np.where(V_P > 0, np.log(V_P + epsilon), np.nan)
    
    # 【核心：防穿越后移】构建目标变量：次日对数波动率
    # 将当天的 sigma_t 向上平移一行 (shift(-1))，作为当天的 target
    df['target_sigma_t_plus_1'] = pd.Series(sigma_t).shift(-1)
    
    # 删除最后一行（因 shift(-1) 产生的 NaN），以及任何因异常价格产生的 NaN
    df = df.dropna(subset=['target_sigma_t_plus_1']).reset_index(drop=True)
    
    return df

# --- 执行主逻辑 ---
# 确保输出目录存在
os.makedirs('../data', exist_ok=True)
os.makedirs('../models', exist_ok=True)
os.makedirs('../results', exist_ok=True)

# 定义文件路径
BASE_DATA_PATH = '../data/merged_daily.csv'
LLM_DATA_PATH = '../data/merged_daily_gemini-1.5-flash_opinion.csv'
OUTPUT_PATH = '../data/01_preprocessed_data.csv'

# 执行合并与预处理
df_merged = load_and_merge_data(BASE_DATA_PATH, LLM_DATA_PATH)
df_final = preprocess_and_build_target(df_merged)

# 导出预处理后的数据
df_final.to_csv(OUTPUT_PATH, index=False)
print(f"\n[成功] 预处理数据已保存至: {OUTPUT_PATH}")

# 打印状态和错位检验
print("\n=== 数据集状态 ===")
print(f"df.shape: {df_final.shape}")

print("\n=== 波动率错位检验 (最后5行) ===")
print(df_final[['timestamp', 'high', 'low', 'target_sigma_t_plus_1']].tail())

当前使用的计算设备: cuda
显卡型号: NVIDIA GeForce RTX 4060 Laptop GPU

[成功] 预处理数据已保存至: ../data/01_preprocessed_data.csv

=== 数据集状态 ===
df.shape: (2337, 36)

=== 波动率错位检验 (最后5行) ===
      timestamp     high      low  target_sigma_t_plus_1
2332 2024-06-24  63397.0  63052.0             -11.273785
2333 2024-06-25  60699.0  60340.0             -12.555604
2334 2024-06-26  61919.0  61726.0             -11.814030
2335 2024-06-27  61112.0  60836.0             -12.184682
2336 2024-06-28  61824.0  61592.0             -11.084055


In [5]:
import pandas as pd

# 定义文件路径（严格遵守相对路径规范）
BASE_DATA_PATH = '../data/merged_daily.csv'
LLM_DATA_PATH = '../data/merged_daily_gemini-1.5-flash_opinion.csv'
FINAL_DATA_PATH = '../data/01_preprocessed_data.csv'

def check_sample_sizes():
    # 1. 获取原始总天数
    try:
        df_base = pd.read_csv(BASE_DATA_PATH)
        df_llm = pd.read_csv(LLM_DATA_PATH)
        original_base_len = len(df_base)
        original_llm_len = len(df_llm)
    except FileNotFoundError as e:
        print(f"找不到原始文件: {e}")
        return

    # 2. 计算合并对齐后的连续交易日 (内连接后的样本量)
    df_base['timestamp'] = pd.to_datetime(df_base['timestamp'])
    df_llm['timestamp'] = pd.to_datetime(df_llm['timestamp'])
    
    # 模拟上一步的 inner join 过程以获取合并后的行数
    df_merged = pd.merge(
        df_base, 
        df_llm[['timestamp', 'reasoning_text', 'sentiment_class', 'action_class', 'action_score']], 
        on='timestamp', 
        how='inner'
    )
    merged_len = len(df_merged)

    # 3. 获取最终有效样本量 (剔除开头NaN、异常值以及末尾因shift产生的NaN后)
    try:
        df_final = pd.read_csv(FINAL_DATA_PATH)
        final_len = len(df_final)
    except FileNotFoundError:
        print("找不到 01_preprocessed_data.csv，请确保上一个 Cell 已经成功运行并保存了文件。")
        return

    # 打印结果，方便填入表格
    print("="*40)
    print("📊 样本量统计结果 (用于填入论文表格)")
    print("="*40)
    print(f"【原始总天数】")
    print(f" - 基础特征数据 (merged_daily.csv): \t{original_base_len} 行")
    print(f" - LLM情绪数据 (gemini_opinion.csv): \t{original_llm_len} 行")
    print("-" * 40)
    print(f"【合并对齐后的连续交易日】: \t\t{merged_len} 行")
    print("-" * 40)
    print(f"【最终有效样本量】")
    print(f" (剔除缺失值及因次日预测对齐产生的无效行): \t{final_len} 行")
    print("="*40)

# 执行统计
check_sample_sizes()

📊 样本量统计结果 (用于填入论文表格)
【原始总天数】
 - 基础特征数据 (merged_daily.csv): 	2338 行
 - LLM情绪数据 (gemini_opinion.csv): 	2338 行
----------------------------------------
【合并对齐后的连续交易日】: 		2338 行
----------------------------------------
【最终有效样本量】
 (剔除缺失值及因次日预测对齐产生的无效行): 	2337 行


In [6]:
import pandas as pd

# ====================================================
# 0. 数据加载 (承接上一步的预处理结果)
# ====================================================
data_path = '../data/01_preprocessed_data.csv'
df = pd.read_csv(data_path)
df['timestamp'] = pd.to_datetime(df['timestamp'])

print(f"已成功加载预处理数据，当前数据形状: {df.shape}")

# ====================================================
#  生成表 3-3：目标变量统计分析 (用于 3.2 节)
# ====================================================
# 获取目标变量序列并计算统计指标
y_target = df['target_sigma_t_plus_1']
stats_df = y_target.describe()

# 计算偏度 (Skewness) 与峰度 (Kurtosis) 以验证正态性/厚尾特征
table_3_3 = pd.DataFrame({
    "统计量": ["均值 (Mean)", "标准差 (Std)", "最小值 (Min)", "25% 分位数", "中位数 (50%)", "75% 分位数", "最大值 (Max)", "偏度 (Skew)", "峰度 (Kurt)"],
    "数值": [
        stats_df['mean'], stats_df['std'], stats_df['min'],
        stats_df['25%'], stats_df['50%'], stats_df['75%'],
        stats_df['max'], y_target.skew(), y_target.kurt()
    ]
}).round(4) # 保留4位小数，符合学术规范

print("\n--- 表 3-3：目标变量 sigma_{t+1} 统计分析 ---")
print(table_3_3.to_string(index=False))


已成功加载预处理数据，当前数据形状: (2337, 36)

--- 表 3-3：目标变量 sigma_{t+1} 统计分析 ---
      统计量       数值
均值 (Mean) -10.7194
标准差 (Std)   1.5724
最小值 (Min) -17.8925
  25% 分位数 -11.6981
中位数 (50%) -10.6739
  75% 分位数  -9.6694
最大值 (Max)  -4.6174
偏度 (Skew)  -0.2164
峰度 (Kurt)   0.5590


In [8]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

# ==========================================
# 0. 全局环境与字体配置 (Windows 中文支持)
# ==========================================
# 设置黑体 (SimHei) 以显示中文，解决负号 '-' 显示为方块的问题
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False
# 设置全局图形美化风格
sns.set_theme(style="whitegrid", font='SimHei', rc={'axes.unicode_minus': False})

# 确保输出文件夹存在
os.makedirs('../results', exist_ok=True)

# 加载上一步处理好的数据
DATA_PATH = '../data/01_preprocessed_data.csv'
try:
    df = pd.read_csv(DATA_PATH)
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    # 提取无缺失值的目标变量用于分布和自相关分析
    target_series = df['target_sigma_t_plus_1'].dropna()
except FileNotFoundError:
    print(f"错误：找不到 {DATA_PATH}，请确保前面的预处理步骤已成功执行。")
    raise

# ==========================================
# 1. 图 3-1：价格+波动率双轴图 
# ==========================================
print("正在生成 图 3-1：价格+波动率双轴图...")
fig1, ax1 = plt.subplots(figsize=(12, 6))

# 绘制左轴 (比特币收盘价)
color1 = 'tab:blue'
ax1.set_xlabel('日期 (Date)', fontsize=12)
ax1.set_ylabel('比特币收盘价 (USD)', color=color1, fontsize=12)
line1 = ax1.plot(df['timestamp'], df['close'], color=color1, label='收盘价 (Close)', linewidth=1.5)
ax1.tick_params(axis='y', labelcolor=color1)
# 旋转X轴日期标签以防重叠
plt.setp(ax1.get_xticklabels(), rotation=45)

# 绘制右轴 (次日对数极值波动率)
ax2 = ax1.twinx()  
color2 = 'tab:orange'
ax2.set_ylabel('次日对数极值波动率 ($\sigma_{t+1}$)', color=color2, fontsize=12)
# 使用较低的透明度 alpha 防止遮挡价格曲线
line2 = ax2.plot(df['timestamp'], df['target_sigma_t_plus_1'], color=color2, label='对数波动率', alpha=0.5, linewidth=1)
ax2.tick_params(axis='y', labelcolor=color2)

# 合并两个轴的图例，并放置在右上角
lines = line1 + line2
labels = [l.get_label() for l in lines]
ax1.legend(lines, labels, loc='upper right')

# 严禁原生标题，保持顶部整洁
ax1.set_title("") 
fig1.tight_layout()
fig1.savefig('../results/3_1_price_volatility.png', dpi=300, format='png', bbox_inches='tight')
plt.close(fig1) # 释放内存


# ==========================================
# 2. 图 3-2：波动率分布直方图 (Histogram + KDE)
# ==========================================
print("正在生成 图 3-2：波动率分布直方图...")
fig2, ax = plt.subplots(figsize=(10, 6))

# 使用 Seaborn 绘制直方图和核密度估计曲线 (KDE)
sns.histplot(target_series, kde=True, bins=50, color='steelblue', edgecolor='black', ax=ax)

ax.set_xlabel('次日对数极值波动率 ($\sigma_{t+1}$)', fontsize=12)
ax.set_ylabel('频数 (Frequency)', fontsize=12)

# 严禁原生标题
ax.set_title("")
fig2.tight_layout()
fig2.savefig('../results/3_2_volatility_distribution.png', dpi=300, format='png', bbox_inches='tight')
plt.close(fig2)


# ==========================================
# 3. 图 3-10：ACF/PACF 图 (Autocorrelation)
# ==========================================
print("正在生成 图 3-10：ACF/PACF 自相关图...")
# 创建上下两个子图
fig3, (ax_acf, ax_pacf) = plt.subplots(2, 1, figsize=(12, 8))

# 绘制 ACF (设置 lags=40 观察长短记忆性)
plot_acf(target_series, ax=ax_acf, lags=40, alpha=0.05, title="")
ax_acf.set_ylabel('自相关系数 (ACF)', fontsize=12)
ax_acf.set_xlabel('') # 上图可省略 X 轴标签

# 绘制 PACF
# 注意：若遇到 "UserWarning: Cannot compute initial kd..." 可通过 method='ywm' 参数缓解，此处保持默认
plot_pacf(target_series, ax=ax_pacf, lags=40, alpha=0.05, title="")
ax_pacf.set_ylabel('偏自相关系数 (PACF)', fontsize=12)
ax_pacf.set_xlabel('滞后阶数 (Lags)', fontsize=12)

# 严禁原生标题 (通过 title="" 已在上文实现，双重保险)
ax_acf.set_title("")
ax_pacf.set_title("")

fig3.tight_layout()
fig3.savefig('../results/3_10_acf_pacf.png', dpi=300, format='png', bbox_inches='tight')
plt.close(fig3)

print("\n[成功] 所有图表已按规范生成并保存至 ../results/ 目录下！")
print("保存的文件名依次为：")
print("1. ../results/3_1_price_volatility.png")
print("2. ../results/3_2_volatility_distribution.png")
print("3. ../results/3_10_acf_pacf.png")

正在生成 图 3-1：价格+波动率双轴图...
正在生成 图 3-2：波动率分布直方图...
正在生成 图 3-10：ACF/PACF 自相关图...

[成功] 所有图表已按规范生成并保存至 ../results/ 目录下！
保存的文件名依次为：
1. ../results/3_1_price_volatility.png
2. ../results/3_2_volatility_distribution.png
3. ../results/3_10_acf_pacf.png
